In [ ]:
import IPython.display as ipd
from IPython.display import display
import soundfile as sf
import matplotlib.pyplot as plt
import librosa
import numpy as np
%config InlineBackend.figure_format = 'retina'
from qwen_tts import Qwen3TTSTokenizer
from qwen_tts.core.tokenizer_12hz.configuration_qwen3_tts_tokenizer_v2 import (
    Qwen3TTSTokenizerV2DecoderConfig,
)
from qwen_tts.core.tokenizer_12hz.modeling_qwen3_tts_tokenizer_v2 import (
    Qwen3TTSTokenizerV2Decoder,
)

from pathlib import Path
from safetensors.torch import load_file

In [ ]:
def plot_waveform(y, sr, title=None):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256, fmax=sr//2)
    fig, ax = plt.subplots()
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, 
                                   x_axis='time',
                                   y_axis='mel', 
                                   sr=sr,
                                   fmax=sr//2, 
                                   ax=ax)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.axhline(12000, color='w', linestyle='--', label='12kHz', alpha=0.2)
    ax.set_ylim(0, 48000//2)
    ax.set_xlabel('Time (s)')
    ax.set(title=f'Mel-frequency spectrogram {"(" + title + ")" if title else ""}')

# Prepare models

In [ ]:
tokenizer_24k = Qwen3TTSTokenizer.from_pretrained("Qwen/Qwen3-TTS-Tokenizer-12Hz")

In [ ]:
def load_tokenizer_for_decoder_block_48k_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    tokenizer_48k = Qwen3TTSTokenizer.from_pretrained("Qwen/Qwen3-TTS-Tokenizer-12Hz")
    decoder = tokenizer_48k.model.decoder

    # configを48kHz用に更新して新デコーダーを作成
    config_dict = decoder.config.to_dict()
    config_dict["upsample_rates"] = [8, 5, 4, 3, 2]
    new_config = Qwen3TTSTokenizerV2DecoderConfig(**config_dict)
    new_decoder = Qwen3TTSTokenizerV2Decoder(new_config)

    # 重みをロード
    new_decoder.load_state_dict(decoder.state_dict(), strict=False)
    checkpoint = load_file(checkpoint_path / "decoder_block.safetensors")
    new_decoder.load_state_dict(checkpoint, strict=False)

    # デコーダーを差し替え
    tokenizer_48k.model.decoder = new_decoder
    tokenizer_48k.model.decode_upsample_rate = 3840
    tokenizer_48k.model.output_sample_rate = 48000
    return tokenizer_48k

In [ ]:
tokenizer_48k = load_tokenizer_for_decoder_block_48k_checkpoint("./output/run2/checkpoint-best")


In [ ]:
type(tokenizer_24k.model.decoder)

In [ ]:
type(tokenizer_48k.model.decoder)

In [ ]:
tokenizer_24k.model.output_sample_rate

In [ ]:
tokenizer_48k.model.output_sample_rate

# Reconstruct

In [ ]:
filename = "../tokenizer48k/audio/daijoubudayosaishokara_02.wav"

In [ ]:
original_wav, original_sr = sf.read(filename, dtype="float32")
display(ipd.Audio(original_wav, rate=original_sr))

In [ ]:
encoded = tokenizer_24k.encode(audios=original_wav, sr=original_sr)

In [ ]:
shape = encoded.audio_codes[0].shape
tokens = shape[0]
approx_bytes = shape[0] * shape[1] * 2
print(f"{len(original_wav)/original_sr:.2f} seconds, {tokens} tokens (approx {approx_bytes} Bytes)")

In [ ]:
reconstruct_24k, sr = tokenizer_24k.decode(encoded)
display(ipd.Audio(reconstruct_24k[0], rate=sr))
sr

In [ ]:
reconstruct_48k, sr = tokenizer_48k.decode(encoded)
display(ipd.Audio(reconstruct_48k[0], rate=sr))
sr

In [ ]:
plot_waveform(original_wav, original_sr, title="Original")

In [ ]:
plot_waveform(reconstruct_24k[0], 24000, title="Reconstructed 24kHz")

In [ ]:
plot_waveform(reconstruct_48k[0], 48000, title="Reconstructed 48kHz")